# A · Agency landscape & trends
**Question:** *How are the funding agencies developing, and how has Texas SBIR/STTR grown?*

Reads the cleaned dataframe from `01_ingest_clean` (`data/tx_sbir_clean.parquet`). Every chart
below is followed by **what it means** and a **methodology & source** note.

> **Source:** SBIR/STTR Award Data (U.S. Small Business Administration), filtered to Texas,
> 2016–2025. See `../SOURCES.md`. **All dollars are nominal (not inflation-adjusted).**

In [ ]:
# --- Setup & house style (Colab-friendly) ---
# !pip install -q pandas numpy pyarrow matplotlib
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Colab-aware paths: use the Drive folder in Colab, fall back to local repo otherwise.
try:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/TX_SBIR_STTR_Analysis'
except Exception:
    BASE = '..'
DATA_DIR = os.environ.get('DATA_DIR', f'{BASE}/data')
OUT = os.environ.get('OUT_DIR', f'{BASE}/outputs'); os.makedirs(OUT, exist_ok=True)
df = pd.read_parquet(os.path.join(DATA_DIR, 'tx_sbir_clean.parquet'))
YR = 'award_year'
years = list(range(int(df[YR].min()), int(df[YR].max())+1))

# House style: quiet grid, no top/right spines, readable sizes.
plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 120, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e6e6e6', 'grid.linewidth': 0.8,
    'axes.axisbelow': True, 'axes.edgecolor': '#888', 'figure.facecolor': 'white',
})
# Fixed, CVD-validated entity colors (order is fixed; 'Other' is neutral gray).
PAL = {'DOD':'#0072B2','HHS':'#E69F00','NASA':'#009E73','NSF':'#CC79A7','DOE':'#D55E00','Other':'#7f7f7f'}
TOP_AGENCIES = ['DOD','HHS','NASA','NSF','DOE']
BLUE, ORANGE = '#0072B2', '#E69F00'
usd = FuncFormatter(lambda v,_: f'${v/1e6:,.0f}M' if v>=1e6 else f'${v:,.0f}')
def save(fig, name):
    fig.savefig(os.path.join(OUT, name), bbox_inches='tight', facecolor='white'); return fig
def agency_bucket(a):
    return a if a in TOP_AGENCIES else 'Other'
df['agency_grp'] = df['agency'].map(agency_bucket)
print(f'Loaded {len(df):,} awards, {df["firm_key"].nunique():,} unique firms, window {years[0]}-{years[-1]}')

## The headline
**Methodology:** totals over the 10-year window; growth compares the last window year to the first.
Growth is nominal dollars, not inflation-adjusted (a limitation stated in `../METHODOLOGY.md`).

In [ ]:
# --- Headline stat tiles ---
tot_awards = len(df)
tot_firms  = df['firm_key'].nunique()
tot_usd    = df['award_amount_num'].sum()
y0, y1 = years[0]-1, years[-1]  # compare last year vs the year before the window (10y earlier)
# growth vs a decade earlier uses the full source file, so reload raw for the base year if present
base = df[df[YR]==years[0]]['award_amount_num'].sum()
last = df[df[YR]==years[-1]]['award_amount_num'].sum()
growth = (last/base - 1) if base else float('nan')
tiles = [(f'${tot_usd/1e9:.2f}B','total SBIR/STTR $'),
         (f'{tot_awards:,}','awards'),
         (f'{tot_firms:,}','unique TX firms'),
         (f'{growth*100:+.0f}%', f'award $ {years[-1]} vs {years[0]}')]
fig, axes = plt.subplots(1, 4, figsize=(11, 1.9))
for ax,(big,lab) in zip(axes, tiles):
    ax.axis('off')
    ax.text(0.5,0.62,big,ha='center',va='center',fontsize=22,fontweight='bold',color='#1a1a1a')
    ax.text(0.5,0.18,lab,ha='center',va='center',fontsize=10,color='#666')
fig.suptitle(f'Texas SBIR/STTR awards, {years[0]}–{years[-1]}', fontsize=13, fontweight='bold', y=1.05)
save(fig,'A0_headline.png'); plt.show()

### What this means
Over 2016–2025, Texas small businesses captured **~$1.8B** across **~3,400** SBIR/STTR awards
to **~880** distinct firms — and annual award dollars **roughly doubled (~+100%)** across the window
(and are up ~+145% vs. 2015, just before it). This is the establishing fact: SBIR/STTR is a large and
*growing* non-dilutive R&D channel for Texas firms.

## Awards & awarded firms per year
**Methodology:** award count = rows per `award_year`; firm count = distinct `firm_key` per year
(dedup on UEI, else normalized name). This mirrors the NC SBTDC chart you flagged.
**Source:** SBA SBIR/STTR award data, TX.

In [ ]:
# --- Awards & awarded firms per year (the NC-style chart you checkmarked) ---
g = df.groupby(YR)
aw = g.size().reindex(years, fill_value=0)
fm = g['firm_key'].nunique().reindex(years, fill_value=0)
x = np.arange(len(years)); w = 0.4
fig, ax = plt.subplots(figsize=(11,4.2))
b1 = ax.bar(x-w/2, aw.values, w, label='Award count', color=BLUE)
b2 = ax.bar(x+w/2, fm.values, w, label='# of firms', color=ORANGE)
ax.bar_label(b1, padding=2, fontsize=8, color='#444')
ax.bar_label(b2, padding=2, fontsize=8, color='#444')
ax.set_xticks(x); ax.set_xticklabels(years)
ax.set_ylabel('Count'); ax.set_ylim(0, max(aw.max(),fm.max())*1.15)
ax.set_title('SBIR/STTR awards and awarded firms in Texas', fontweight='bold', loc='left')
ax.legend(frameon=False, loc='upper left')
save(fig,'A1_awards_firms.png'); plt.show()

### What this means
Both awards and the number of funded firms rose from ~2016, **peaking around 2021–2022 (~400+
awards/yr)** before easing in 2024–2025. The gap between the blue and orange bars shows that many
firms win **more than one** award in a year — a repeat-winner dynamic we quantify in Module B.

## Total award dollars per year
**Methodology:** sum of `award_amount_num` per year. **Source:** SBA SBIR/STTR award data, TX.

In [ ]:
# --- Total award dollars per year (magnitude over time) ---
dpy = df.groupby(YR)['award_amount_num'].sum().reindex(years, fill_value=0)
fig, ax = plt.subplots(figsize=(11,4))
bars = ax.bar(range(len(years)), dpy.values, color=BLUE, width=0.7)
ax.bar_label(bars, labels=[f'${v/1e6:.0f}M' for v in dpy.values], padding=2, fontsize=8, color='#444')
ax.set_xticks(range(len(years))); ax.set_xticklabels(years)
ax.yaxis.set_major_formatter(usd); ax.set_ylabel('Award dollars')
ax.set_title('Total SBIR/STTR award dollars per year, Texas', fontweight='bold', loc='left')
save(fig,'A2_dollars_year.png'); plt.show()

### What this means
Dollars climbed faster than award *counts*, because average award size rose (more Phase II money —
see the phase chart). The funding pipeline into Texas roughly **doubled** over the decade.

## Awards by funding agency over time
**Methodology:** awards per year split by agency; the five largest agencies are shown, all others
folded into a neutral **Other**. Colors are fixed per agency. **Source:** SBA SBIR/STTR award data, TX.

In [ ]:
# --- Awards by agency over time (how the agencies are developing) ---
piv = (df.groupby([YR,'agency_grp']).size().unstack(fill_value=0).reindex(years, fill_value=0))
order = [a for a in TOP_AGENCIES+['Other'] if a in piv.columns]
fig, ax = plt.subplots(figsize=(11,4.6))
for a in order:
    ax.plot(years, piv[a].values, marker='o', ms=4, lw=2.2, color=PAL[a], label=a)
# de-collide the right-edge labels: enforce a minimum vertical gap
ymax = float(piv[order].values.max()); gap = ymax*0.06
placed = []
for a, val in sorted(((a, float(piv[a].values[-1])) for a in order), key=lambda t: t[1]):
    y = val if not placed or val-placed[-1][1] >= gap else placed[-1][1]+gap
    placed.append((a, y))
for a, y in placed:
    ax.annotate(a, (years[-1], y), xytext=(8,0), textcoords='offset points',
                va='center', fontsize=9, fontweight='bold', color=PAL[a])
ax.set_xticks(years); ax.set_ylabel('Awards per year')
ax.set_title('SBIR/STTR awards by funding agency, Texas', fontweight='bold', loc='left')
ax.margins(x=0.02); ax.set_xlim(years[0], years[-1]+0.8)
save(fig,'A3_agency_lines.png'); plt.show()

### What this means
**DoD is the engine of Texas SBIR/STTR and it grew the most.** HHS (NIH) is a distant but steady
second; NASA, DOE, and NSF are meaningful but much smaller. So “how the agencies are developing” in
Texas is largely a DoD story — which shapes the *kind* of company that wins here (defense-adjacent tech).

## Share of dollars by agency
**Methodology:** total `award_amount_num` by agency over the window, as a share of all TX dollars.
**Source:** SBA SBIR/STTR award data, TX.

In [ ]:
# --- Share of dollars by agency over the window ---
share = df.groupby('agency_grp')['award_amount_num'].sum().sort_values()
share = share.reindex([a for a in ['Other','DOE','NSF','NASA','HHS','DOD'] if a in share.index])
fig, ax = plt.subplots(figsize=(9,3.6))
bars = ax.barh(share.index, share.values, color=[PAL[a] for a in share.index])
tot = share.sum()
for b,v in zip(bars, share.values):
    ax.text(v+tot*0.01, b.get_y()+b.get_height()/2, f'${v/1e6:.0f}M ({100*v/tot:.0f}%)',
            va='center', fontsize=9, color='#333')
ax.xaxis.set_major_formatter(usd); ax.set_xlim(0, tot*0.72)
ax.set_title(f'Share of SBIR/STTR dollars by agency, {years[0]}–{years[-1]}', fontweight='bold', loc='left')
save(fig,'A4_agency_share.png'); plt.show()

### What this means
**DoD ~56% and HHS ~25% of dollars — together ~81%.** This matches the national pattern (DoD + HHS
fund the large majority of the whole program) and confirms Texas is concentrated in the two mega-agencies.

## SBIR vs STTR
**Methodology:** dollars per year split by program. STTR requires a research-institution partner.
**Source:** SBA SBIR/STTR award data, TX.

In [ ]:
# --- SBIR vs STTR dollars per year ---
pp = (df.groupby([YR,'program_norm'])['award_amount_num'].sum().unstack(fill_value=0).reindex(years, fill_value=0))
fig, ax = plt.subplots(figsize=(11,4))
for prog,color in [('SBIR', BLUE), ('STTR', ORANGE)]:
    if prog in pp.columns:
        ax.plot(years, pp[prog].values, marker='o', ms=4, lw=2.4, color=color, label=prog)
        ax.annotate(prog,(years[-1],pp[prog].values[-1]),xytext=(6,0),textcoords='offset points',
                    va='center',fontsize=9,fontweight='bold',color=color)
ax.set_xticks(years); ax.yaxis.set_major_formatter(usd); ax.set_ylabel('Award dollars')
ax.set_xlim(years[0], years[-1]+0.8)
ax.set_title('SBIR vs STTR award dollars per year, Texas', fontweight='bold', loc='left')
save(fig,'A5_sbir_sttr.png'); plt.show()

### What this means
**SBIR dominates (~86% of dollars); STTR (~14%)** is the smaller university-partnered track. If growing
university–industry tech transfer is a goal, STTR is the lever with the most headroom.

## Phase I vs Phase II — awards vs. dollars
**Methodology:** among Phase I/II awards, compare each phase's share of *award count* to its share of
*dollars*. **Source:** SBA SBIR/STTR award data, TX. (Phase III carries no SBIR dollars and is analyzed
separately in Module E.)

In [ ]:
# --- Phase I vs Phase II: share of awards vs share of dollars ---
ph = df[df['phase_norm'].isin(['Phase I','Phase II'])]
cnt = ph.groupby('phase_norm').size(); dol = ph.groupby('phase_norm')['award_amount_num'].sum()
cnt_s = 100*cnt/cnt.sum(); dol_s = 100*dol/dol.sum()
labels=['Phase I','Phase II']; x=np.arange(2); w=0.38
fig, ax = plt.subplots(figsize=(7.5,4))
b1=ax.bar(x-w/2,[cnt_s.get(l,0) for l in labels],w,label='% of awards',color='#9aa0a6')
b2=ax.bar(x+w/2,[dol_s.get(l,0) for l in labels],w,label='% of dollars',color=BLUE)
ax.bar_label(b1,fmt='%.0f%%',padding=2,fontsize=9); ax.bar_label(b2,fmt='%.0f%%',padding=2,fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel('Percent'); ax.set_ylim(0,100)
ax.set_title('Where the awards are vs. where the money is', fontweight='bold', loc='left')
ax.legend(frameon=False)
save(fig,'A6_phase_mix.png'); plt.show()

### What this means
**Phase I is most of the *awards* (~66%) but Phase II is most of the *money* (~78%)** — because Phase II
awards are ~7–10× larger. This is the funnel in miniature: many small feasibility bets, fewer but much
bigger development bets. Whether those Phase II bets convert to commercialization is Module E.

---
*Next: **Module B · Who they invest in** — top firms, repeat winners (e.g. one TX firm holds 300+
awards), first-time vs. repeat awardees, and demographic breakdowns (woman-owned, HUBZone,
disadvantaged). Geography (Module D) unlocks once the county crosswalk source is approved.*